In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lfreedom2750","key":"fb46b035b65134128288a6ce5f370912"}'}

In [ ]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-train-data-v2
!unzip -q fakeface-train-data-v2.zip -d train_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-train-data-v2
License(s): unknown
 98% 2.17G/2.21G [00:07<00:00, 555MB/s]
100% 2.21G/2.21G [00:07<00:00, 309MB/s]


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-valid-data-v2
!unzip -q fakeface-valid-data-v2.zip -d valid_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-valid-data-v2
License(s): unknown
 88% 653M/745M [00:00<00:00, 1.31GB/s]
100% 745M/745M [00:00<00:00, 1.32GB/s]


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-test-data-v2
!unzip -q fakeface-test-data-v2.zip -d test_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-test-data-v2
License(s): unknown
 87% 650M/747M [00:00<00:00, 1.29GB/s]
100% 747M/747M [00:00<00:00, 1.30GB/s]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image

In [ ]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/content/train_dataset", transform=transform)
val_dataset   = datasets.ImageFolder("/content/valid_dataset", transform=transform)
test_dataset = datasets.ImageFolder("/content/test_dataset", transform=transform)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader   = DataLoader(test_dataset, batch_size=256, shuffle=False,num_workers=4, pin_memory=True)

In [ ]:
from torch.nn import functional as F
import torch
import torch.nn as nn
from torchvision import models

class MobileNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[-1].in_features
        self.model.classifier[-1] = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch.nn as nn
from torchvision import models

class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = MobileNetV3()
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

epoch_times = []
start_training = time.time()

for epoch in range(10):
    start_epoch = time.time()

    model.train()
    total_loss = 0
    total_batches = len(train_loader)

    for x, y in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False):
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y)
        print(f"Loss: {loss.item():.4f}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg = total_loss / len(train_loader)

    model.eval()
    correct, total = 0, 0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)

    print(f"[MobileNetV3] Epoch {epoch+1} | Train Loss: {avg:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {epoch_time:.2f}s")

total_time = time.time() - start_training
avg_time = sum(epoch_times) / len(epoch_times)

print(f"\nTotal training time: {total_time:.2f}s")
print(f"Average time per epoch: {avg_time:.2f}s")

torch.save(model.state_dict(), "mobilenetv3.pth")

cuda


[Epoch 1] Training:   0%|          | 0/473 [00:00<?, ?it/s]

Loss: 0.7115


[Epoch 1] Training:   0%|          | 2/473 [00:03<11:35,  1.48s/it]

Loss: 0.7141
Loss: 0.6968


...

Loss: 0.0010
Loss: 0.0064


[Epoch 10] Training: 100%|█████████▉| 471/473 [01:18<00:00,  6.11it/s]

Loss: 0.0019
Loss: 0.0024


Loss: 0.0078


[MobileNetV3] Epoch 10 | Train Loss: 0.0068 | Val Acc: 0.9424 | Val AUC: 0.9865 | Time: 94.24s

Total training time: 945.57s
Average time per epoch: 94.56s


In [ ]:
from sklearn.metrics import classification_report

def evaluate_teacher(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.cpu().numpy()
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
evaluate_teacher(model, test_loader)


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9512    0.9384    0.9448     20000
           1     0.9392    0.9519    0.9455     20000

    accuracy                         0.9452     40000
   macro avg     0.9452    0.9451    0.9451     40000
weighted avg     0.9452    0.9452    0.9451     40000



In [ ]:
import torch

# Load the student model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student = MobileNetV3()  # Instantiate the model again
student.load_state_dict(torch.load("/content/mobilenetv3.pth", map_location=device))

student = student.to(device)

# If you used DataParallel during training, wrap the model again
if torch.cuda.device_count() > 1:
    student = torch.nn.DataParallel(student)

# Count the number of parameters
total_params = sum(p.numel() for p in student.parameters())

print(f"Số lượng tham số của mô hình student (MobileNetV3): {total_params}")

Số lượng tham số của mô hình student (MobileNetV3): 1519906


In [ ]:
class MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch

# Load the student model
student = MobileNetV2()  # Instantiate the model again
student.load_state_dict(torch.load("mobilenet_v2_student_kd.pth", map_location=device))
student = student.to(device)

# If you used DataParallel during training, wrap the model again
if torch.cuda.device_count() > 1:
    student = torch.nn.DataParallel(student)

# Count the number of parameters
total_params = sum(p.numel() for p in student.parameters())

trainable_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f"Số lượng tham số của mô hình student (MobileNetV2): {trainable_params}")

Số lượng tham số của mô hình student (MobileNetV2): 2226434


In [ ]:
class DistillLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.7):
        super().__init__()
        self.T = T
        self.alpha = alpha
        self.kld = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, true_labels):
        kd = self.kld(F.log_softmax(student_logits / self.T, dim=1),
                      F.softmax(teacher_logits / self.T, dim=1)) * (self.T ** 2)
        ce = self.ce(student_logits, true_labels)
        return self.alpha * kd + (1 - self.alpha) * ce

In [ ]:
teacher = MobileNetV3()
teacher.load_state_dict(torch.load("/content/mobilenetv3.pth", map_location=device))
teacher = teacher.to(device)
teacher.eval()

student = MobileNetV2()
student = student.to(device)

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPU với DataParallel.")
    student = nn.DataParallel(student)
    teacher = nn.DataParallel(teacher)

kd_loss_fn = DistillLoss(T=4.0, alpha=0.7)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

def evaluate_model_on_validation(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = float('nan')

    return acc, auc

In [ ]:
import time
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

def train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10):
    ce_loss_fn = nn.CrossEntropyLoss()
    start_training = time.time()

    for epoch in range(epochs):
        student.train()
        teacher.eval()

        total_kd_loss = 0
        total_student_loss = 0
        total_teacher_loss = 0

        all_probs = []
        all_labels = []

        start_epoch = time.time()

        progress_bar = tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False)

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                t_logits = teacher(x)
                teacher_ce_loss = ce_loss_fn(t_logits, y)

            s_logits = student(x)

            student_ce_loss = ce_loss_fn(s_logits, y)
            kd_loss = kd_loss_fn(s_logits, t_logits, y)

            kd_loss.backward()
            optimizer.step()

            total_kd_loss += kd_loss.item()
            total_student_loss += student_ce_loss.item()
            total_teacher_loss += teacher_ce_loss.item()

            probs = torch.softmax(s_logits, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "KD": f"{kd_loss.item():.4f}",
                "StudentCE": f"{student_ce_loss.item():.4f}"
            })

        try:
            train_auc = roc_auc_score(all_labels, all_probs)
        except:
            train_auc = float('nan')

        epoch_time = time.time() - start_epoch

        val_acc, val_auc = evaluate_model_on_validation(student, val_loader, device)

        print(f"[Epoch {epoch+1}] "
              f"KD Loss: {total_kd_loss / len(train_loader):.4f} | "
              f"Student CE Loss: {total_student_loss / len(train_loader):.4f} | "
              f"Teacher CE Loss: {total_teacher_loss / len(train_loader):.4f} | "
              f"Train AUC: {train_auc:.4f} | "
              f"Val AUC: {val_auc:.4f} | Val ACC: {val_acc:.4f} | "
              f"Time: {epoch_time:.2f}s")

    total_time = time.time() - start_training
    print(f"⏱️ Tổng thời gian training: {total_time:.2f} giây ({total_time/60:.2f} phút)")


In [ ]:
from sklearn.metrics import classification_report
import torch

def evaluate_models_report(student, teacher, dataloader, device):
    student.eval()
    teacher.eval()

    all_labels = []
    student_preds = []
    teacher_preds = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            s_logits = student(x)
            t_logits = teacher(x)

            student_cls = torch.argmax(s_logits, dim=1).cpu().numpy()
            teacher_cls = torch.argmax(t_logits, dim=1).cpu().numpy()
            true_labels = y.cpu().numpy()

            student_preds.extend(student_cls)
            teacher_preds.extend(teacher_cls)
            all_labels.extend(true_labels)

    print("📘 [Student Model] Classification Report:")
    print(classification_report(all_labels, student_preds, digits=4))

    print("📗 [Teacher Model] Classification Report:")
    print(classification_report(all_labels, teacher_preds, digits=4))


In [ ]:
train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10)

[Epoch 1] KD Loss: 1.7061 | Student CE Loss: 0.2802 | Teacher CE Loss: 0.0039 | Train AUC: 0.9784 | Val AUC: 0.9904 | Val ACC: 0.9508 | Time: 325.68s


[Epoch 2] KD Loss: 0.5620 | Student CE Loss: 0.0528 | Teacher CE Loss: 0.0039 | Train AUC: 0.9988 | Val AUC: 0.9945 | Val ACC: 0.9644 | Time: 325.33s


[Epoch 3] KD Loss: 0.3648 | Student CE Loss: 0.0167 | Teacher CE Loss: 0.0039 | Train AUC: 0.9998 | Val AUC: 0.9952 | Val ACC: 0.9627 | Time: 325.43s


[Epoch 4] KD Loss: 0.2921 | Student CE Loss: 0.0096 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9951 | Val ACC: 0.9633 | Time: 325.36s


[Epoch 5] KD Loss: 0.2508 | Student CE Loss: 0.0083 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9955 | Val ACC: 0.9700 | Time: 325.42s


[Epoch 6] KD Loss: 0.2186 | Student CE Loss: 0.0072 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9957 | Val ACC: 0.9696 | Time: 325.50s


[Epoch 7] KD Loss: 0.2022 | Student CE Loss: 0.0083 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9960 | Val ACC: 0.9720 | Time: 325.42s


[Epoch 8] KD Loss: 0.1919 | Student CE Loss: 0.0085 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9954 | Val ACC: 0.9698 | Time: 325.42s


[Epoch 9] KD Loss: 0.1748 | Student CE Loss: 0.0086 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9960 | Val ACC: 0.9717 | Time: 325.37s


[Epoch 10] KD Loss: 0.1684 | Student CE Loss: 0.0085 | Teacher CE Loss: 0.0039 | Train AUC: 1.0000 | Val AUC: 0.9962 | Val ACC: 0.9720 | Time: 325.43s
⏱️ Tổng thời gian training: 3552.54 giây (59.21 phút)


In [ ]:
evaluate_models_report(student, teacher, test_loader, device)

📘 [Student Model] Classification Report:
              precision    recall  f1-score   support

           0     0.9696    0.9748    0.9722     20000
           1     0.9747    0.9695    0.9721     20000

    accuracy                         0.9721     40000
   macro avg     0.9721    0.9721    0.9721     40000
weighted avg     0.9721    0.9721    0.9721     40000

📗 [Teacher Model] Classification Report:
              precision    recall  f1-score   support

           0     0.9512    0.9384    0.9448     20000
           1     0.9392    0.9519    0.9455     20000

    accuracy                         0.9452     40000
   macro avg     0.9452    0.9451    0.9451     40000
weighted avg     0.9452    0.9452    0.9451     40000



In [ ]:
torch.save(student.state_dict(), "mobilenet_v2_student_kd.pth")